In [2]:
import h5py
import matplotlib.pyplot as plt
import numpy as np
import os
from netCDF4 import Dataset
from regrid import regrid
import pandas as pd
import cartopy
import cartopy.crs as ccrs
import tqdm
import pickle
import itertools

/home/robbie/anaconda3/envs/algae/lib/python3.13/site-packages/pyproj/crs/crs.py:141: FutureWarning: '+init=<authority>:<code>' syntax is deprecated. '<authority>:<code>' is the preferred initialization method. When making the change, be mindful of axis order changes: https://pyproj4.github.io/pyproj/stable/gotchas.html#axis-order-changes-in-proj-6
  in_crs_string = _prepare_from_proj_string(in_crs_string)


In [7]:
d = Dataset('inputs/sit_dists_array.nc')
d

<class 'netCDF4.Dataset'>
root group (NETCDF4 data model, file format HDF5):
    dimensions(sizes): x(896), y(608), d(15)
    variables(dimensions): float64 latitude(x, y), float64 longitude(x, y), float32 thicknesses(d), float64 sit_dists(x, y, d)
    groups: 

# Calculate fractional transmission after snow albedo effect for all combinations of snow and ice thickness.

In [2]:
def get_light2(snow_depth,
              ice_thickness,
              surface_scattering_layer_albedo = 0.5
                ):
    
    """Gets light transmittance through snow of a given depth
    
        Takes incident light (in this case 100 to give % remaining)
        
        Snow depth and ice thickness in metres.
        
        Works using Beer-Lambert and albedo values from Abraham et al. 2015 JGRO"""

    E_s = 14
    E_i = 1.2
    

    incident_ice = np.exp(-E_s*snow_depth)
    
    trans_ice = incident_ice * (1 - surface_scattering_layer_albedo)
    
    incident_ice_bottom = trans_ice*np.exp(-E_i*ice_thickness)
    
    penetrated = incident_ice_bottom
    
    ################################################
    
    return penetrated

In [3]:
floating_point_factor=fpl=10000

snow_depths = np.arange(0,1.01,0.01)
sits = np.arange(0,21,0.05)
prod=itertools.product(snow_depths,sits)
pens = [get_light2(snow,ice) for snow,ice in prod]
combo=[(int(i*fpl),int(j*fpl)) for i,j in itertools.product(snow_depths,sits)]
depths_iter=[i for i,j in itertools.product(snow_depths,sits)]
sits_iter=[j for i,j in itertools.product(snow_depths,sits)]

In [4]:
pen_df = pd.DataFrame({'combo':combo,'penetration':pens,'depth':depths_iter,'sit':sits_iter})
pen_df.set_index('combo',inplace=True)

In [5]:
pen_df.to_csv('inputs/pen_df.csv')
pen_df = pd.read_csv('inputs/pen_df.csv',index_col='combo')

In [6]:
d = np.array(Dataset('inputs/ssrd.nc')['time'])

/tmp/ipykernel_42266/3750006770.py:1: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  d = np.array(Dataset('inputs/ssrd.nc')['time'])
